In [1]:
%load_ext autoreload
%autoreload 2
import scMPRAforge as scm


2025-08-19 12:58:26.312988: I tensorflow/core/util/util.cc:169] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-08-19 12:58:26.317170: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /vast/palmer/apps/avx2/software/code-server/4.91.1/lib:/vast/palmer/apps/avx2/software/gettext/0.22.5-GCCcore-13.3.0/lib:/vast/palmer/apps/avx2/software/libiconv/1.17-GCCcore-13.3.0/lib:/vast/palmer/apps/avx2/software/ncurses/6.5-GCCcore-13.3.0/lib:/vast/palmer/apps/avx2/software/libxml2/2.12.7-GCCcore-13.3.0/lib:/vast/palmer/apps/avx2/software/XZ/5.4.5-GCCcore-13.3.0/lib:/vast/palmer/apps/avx2/software/expat/2.6.2-GCCcore-13.3.0/lib:/vast/palmer/apps/av

In [2]:
import pandas as pd

## Test wald test

In [3]:

#create dask cluster
from dask.distributed import Client, LocalCluster
cluster=LocalCluster(n_workers=10)
client = Client(cluster)

path="/gpfs/gibbs/pi/reilly/tabula_data/shendure"
name="ortho_primordial"

import os
if os.path.isdir(path+"/"+name):
    print("[+] Model found. Loading...")
    primordial=scm.ortho.load(client,path,name)
    shendure=primordial.training_data
    primordial.precompute_wald(client)
    # primordial.save(path, name) 
else:
    print("[+] Model not found. Creating...")

    #load data
    data_root="/gpfs/gibbs/pi/reilly/tabula_data"
    shendure=scm.scMPRA_data.from_tsv(f"{data_root}/shendure/shendure_counts_grouped.txt")
    
    shendure.set_negative_controls(["minP","noP"])
    shendure.set_reference_cell("Pluripotent")
    shendure.ortho_filter()

    primordial=scm.ortho()
    primordial.criss_cross(client=client,
                       dat=shendure)
    primordial.extract_params(client)
    primordial.precompute_wald(client)  
    primordial.save(path,name)

[+] Model found. Loading...


In [4]:

# Wald tests now hit the precomp cache automatically:
test_fn = scm.build_wald_test_fn(primordial, shendure, cre_mode="vs_reference")
runner  = scm.HypothesisTester(test_fn=test_fn, test_type_name="wald")


In [5]:
# by-cell-type: test many CREs in NeuroectodermBrain vs the 'reference' negative control
hs_ct = scm.make_by_celltype_hypotheses(
    comparison_cell_type="NeuroectodermBrain",
    counts=shendure,
    comparison_cres="all",          # or a list like ["CRE1","CRE2",...]
    reference_cre="reference",      # this is how you labeled minP/noP
    meta="emvar_screen"
)

# by-CRE: test CRE123 across all cell types vs the baseline cell type
hs_cre = scm.make_by_cre_hypotheses(
    comparison_cre="all",
    counts=shendure,
    comparison_cell_types="all",    # or a list
    reference_cell_type="reference",   # will default to counts.reference_cell_type if set
    meta="cell_specificity"
)

# all CREs within each cell type, vs the 'reference' negative control
hs_all_ct = scm.make_all_by_celltype_hypotheses(
    counts=shendure,
    reference_cre="reference",
    meta="emvar_screen",
)

# all cell types for each CRE, vs the dataset’s baseline cell type
hs_all_cre = scm.make_all_by_cre_hypotheses(
    counts=shendure,
    reference_cell_type="reference",  # will be normalized to 'reference'
    meta="cell_specificity",
)




In [6]:
hs_all_ct.to_dataframe()

,comparison_CRE,comparison_cell_type,reference_CRE,reference_cell_type,meta
0,Bend5_chr4_8175,Cardiomyocytes,reference,Cardiomyocytes,emvar_screen
1,Cdk5r1_chr11_12559,Cardiomyocytes,reference,Cardiomyocytes,emvar_screen
2,Col1a1_chr11_15322,Cardiomyocytes,reference,Cardiomyocytes,emvar_screen
3,Col1a2_chr6_77,Cardiomyocytes,reference,Cardiomyocytes,emvar_screen
4,Igfbp4_chr11_16711,Cardiomyocytes,reference,Cardiomyocytes,emvar_screen
...,...,...,...,...,...
1446,Txndc12_chr4_7973,reference,reference,reference,emvar_screen
1447,Lama1_chr17_7787,reference,reference,reference,emvar_screen
1448,Epas1_chr17_10064,reference,reference,reference,emvar_screen
1449,Btg1_chr10_9578,reference,reference,reference,emvar_screen


In [7]:
hs_all_cre.to_dataframe()

,comparison_CRE,comparison_cell_type,reference_CRE,reference_cell_type,meta
0,Bend5_chr4_8168,NeuroectodermBrain,Bend5_chr4_8168,reference,cell_specificity
1,Bend5_chr4_8168,ExEndodermParietal,Bend5_chr4_8168,reference,cell_specificity
2,Bend5_chr4_8168,EpiblastPrimitiveStreak,Bend5_chr4_8168,reference,cell_specificity
3,Bend5_chr4_8168,SurfaceEctoderm,Bend5_chr4_8168,reference,cell_specificity
4,Bend5_chr4_8170,EpiblastPrimitiveStreak,Bend5_chr4_8170,reference,cell_specificity
...,...,...,...,...,...
1240,ubcP,NeuroectodermRostral,ubcP,reference,cell_specificity
1241,ubcP,ExEndodermParietal,ubcP,reference,cell_specificity
1242,ubcP,Haematoendothelial,ubcP,reference,cell_specificity
1243,ubcP,Cardiomyocytes,ubcP,reference,cell_specificity


In [10]:
wald_by_cre = runner.run(hs_all_cre).to_dataframe()

IndexError: only integers, slices (`:`), ellipsis (`...`), numpy.newaxis (`None`) and integer or boolean arrays are valid indices

In [13]:
wald_by_cre

,comparison_CRE,comparison_cell_type,reference_CRE,reference_cell_type,meta,test_statistic,p_value,fold_change,flattened,test_type,bh_p
0,Bend5_chr4_8168,NeuroectodermBrain,Bend5_chr4_8168,reference,cell_specificity,NaN,NaN,NaN,False,wald,1.0
1,Bend5_chr4_8168,ExEndodermParietal,Bend5_chr4_8168,reference,cell_specificity,NaN,NaN,NaN,False,wald,1.0
2,Bend5_chr4_8168,EpiblastPrimitiveStreak,Bend5_chr4_8168,reference,cell_specificity,NaN,NaN,NaN,False,wald,1.0
3,Bend5_chr4_8168,SurfaceEctoderm,Bend5_chr4_8168,reference,cell_specificity,NaN,NaN,NaN,False,wald,1.0
4,Bend5_chr4_8170,EpiblastPrimitiveStreak,Bend5_chr4_8170,reference,cell_specificity,NaN,NaN,NaN,False,wald,1.0
...,...,...,...,...,...,...,...,...,...,...,...
1240,ubcP,NeuroectodermRostral,ubcP,reference,cell_specificity,NaN,NaN,NaN,False,wald,1.0
1241,ubcP,ExEndodermParietal,ubcP,reference,cell_specificity,NaN,NaN,NaN,False,wald,1.0
1242,ubcP,Haematoendothelial,ubcP,reference,cell_specificity,NaN,NaN,NaN,False,wald,1.0
1243,ubcP,Cardiomyocytes,ubcP,reference,cell_specificity,NaN,NaN,NaN,False,wald,1.0


In [8]:
wald_by_ct  = runner.run(hs_all_ct).to_dataframe()

In [9]:
wald_by_ct

,comparison_CRE,comparison_cell_type,reference_CRE,reference_cell_type,meta,test_statistic,p_value,fold_change,flattened,test_type,bh_p
0,Bend5_chr4_8175,Cardiomyocytes,reference,Cardiomyocytes,emvar_screen,10.714203,0.000000e+00,23.673172,False,wald,0.0
1,Cdk5r1_chr11_12559,Cardiomyocytes,reference,Cardiomyocytes,emvar_screen,6.058773,1.371636e-09,4.730881,False,wald,0.0
2,Col1a1_chr11_15322,Cardiomyocytes,reference,Cardiomyocytes,emvar_screen,2.119463,3.405135e-02,2.571112,False,wald,0.0
3,Col1a2_chr6_77,Cardiomyocytes,reference,Cardiomyocytes,emvar_screen,-0.268368,7.884163e-01,0.879807,False,wald,0.0
4,Igfbp4_chr11_16711,Cardiomyocytes,reference,Cardiomyocytes,emvar_screen,0.883484,3.769747e-01,1.554059,False,wald,0.0
...,...,...,...,...,...,...,...,...,...,...,...
1446,Txndc12_chr4_7973,reference,reference,reference,emvar_screen,29.942444,0.000000e+00,11.885299,False,wald,0.0
1447,Lama1_chr17_7787,reference,reference,reference,emvar_screen,-5.679197,1.353289e-08,0.133550,False,wald,0.0
1448,Epas1_chr17_10064,reference,reference,reference,emvar_screen,-4.859352,1.177709e-06,0.326999,False,wald,0.0
1449,Btg1_chr10_9578,reference,reference,reference,emvar_screen,30.837136,0.000000e+00,38.551599,False,wald,0.0


In [13]:
some_ct = next(iter(primordial.by_cell_type.model.keys()))
Xnames = primordial.by_cell_type_design[some_ct].result()["nb_regressors"].columns.tolist()
print(Xnames[:10])

['Intercept', "C(cre_id, contr.treatment(base='reference'))[T.Bend5_chr4_8168]", "C(cre_id, contr.treatment(base='reference'))[T.Bend5_chr4_8170]", "C(cre_id, contr.treatment(base='reference'))[T.Bend5_chr4_8172]", "C(cre_id, contr.treatment(base='reference'))[T.Bend5_chr4_8174]", "C(cre_id, contr.treatment(base='reference'))[T.Bend5_chr4_8175]", "C(cre_id, contr.treatment(base='reference'))[T.Bend5_chr4_8179]", "C(cre_id, contr.treatment(base='reference'))[T.Bend5_chr4_8192]", "C(cre_id, contr.treatment(base='reference'))[T.Bend5_chr4_8199]", "C(cre_id, contr.treatment(base='reference'))[T.Bend5_chr4_8201]"]


In [18]:
primordial.wald_precomp.by_cell_type['Mesoderm'].result()

In [19]:
cluster.close()

2025-08-19 12:54:47,486 - distributed.scheduler - WARNING - Removing worker 'tcp://127.0.0.1:38321' caused the cluster to lose already computed task(s), which will be recomputed elsewhere: {'_build_wald_precomp_for_subset-16fba006802d086593596018d0b91390', 'lambda-3c5c6626eb14a21aa4c71d87f98bde82', '_build_wald_precomp_for_subset-bb53548833ee55a0fe2470741b529906', '_build_wald_precomp_for_subset-1bdbc9180eb70d0fcd525443d2dd505b', 'lambda-170cc010714afc9234ea68b7ae038e8f', 'lambda-1da1de772ff579bb0081dbf162129df9', 'lambda-227a306833d4432ba1173657f46f4ee4', 'lambda-43698d5f58f7120676089f39e3c39d9e', 'lambda-7f73e0dd085a2b415f5091783f258f6a', '_build_wald_precomp_for_subset-e359e60f52d4efc7dade4b7626966a0d', 'lambda-f5e78d3fb0303ceb707c08e5f3195bd0', '_build_wald_precomp_for_subset-1d922d7e2cc287845c2943f8d29b1213', 'lambda-4b8009dfcd8b45ef1d2e9aefce48d5d1', 'lambda-662c8e8f53f0d7e231ec8a79ee810360', 'lambda-f2d04477ba05dbb14908260904a3ec82', 'lambda-7458a8d97b427a3141d6b9bde008e622', 'l

In [10]:
import numpy as np

In [13]:
np.exp(3.1643424)

23.673171435586262